# AI651 PA3 — Spatio-Temporal GNN Forecasting on LargeST-SD

One notebook for Questions 1–7. It is written for **Google Colab** and runs top to bottom with *Runtime → Run all*.

**Before the real run**
1. Set `SMOKE_TEST = True` and run all cells once (≈2–5 min). This builds small synthetic files in the same format as the real data and runs every step end to end, so problems show up before the 2.3 GB download.
2. Set `SMOKE_TEST = False`, then *Runtime → Restart session and run all*.

**Switches** (next cell)
- `USE_DRIVE` — `False` (default): everything stays on Colab's temporary disk. `True`: mounts Google Drive and caches the processed San Diego data (~100 MB) and the results in `MyDrive/DLSTG_PA3`, which survives disconnects.
- `RESUME` — when a model's checkpoint and metrics already exist, they are loaded instead of retrained.

**Getting the results out**: the last cell zips `results/`, `figures/` and `checkpoints/` into `pa3_outputs.zip` and downloads it. Also save this notebook with its outputs via *File → Download → .ipynb*.

**Data**: LargeST (Liu et al., NeurIPS 2023 Datasets & Benchmarks), licence CC BY-NC 4.0. It is downloaded from the Hugging Face copy used by Torch Spatiotemporal (`tsl`), so no Kaggle account is needed: `2020/data.h5` (1.16 GB), `2021/data.h5` (1.17 GB), `sensors.zip` (0.7 MB).

**Code this notebook follows or adapts** (links pinned to the commit that was read):
- LargeST, San Diego subset = `District == 11`, adjacency indexed with the same ordered list: [`data/sd/generate_sd_dataset.ipynb`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/data/sd/generate_sd_dataset.ipynb)
- LargeST, 15-minute resampling (`resample('15T').mean().round(0)` then `fillna(0)`): [`data/ca/process_ca_his.ipynb`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/data/ca/process_ca_his.ipynb)
- LargeST, input/target offsets: [`data/generate_data_for_training.py#L51-L52`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/data/generate_data_for_training.py#L51-L52)
- LargeST, symmetric / row adjacency normalisation (`calculate_sym_adj`, `calculate_asym_adj`): [`src/utils/graph_algo.py#L53-L70`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/utils/graph_algo.py#L53-L70)
- LargeST, masked MAE/RMSE: [`src/utils/metrics.py`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/utils/metrics.py)
- Graph WaveNet adaptive adjacency `softmax(relu(E1 E2ᵀ))`, as in LargeST's copy: [`src/models/gwnet.py#L17-L18`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/models/gwnet.py#L17-L18) and [`#L79`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/models/gwnet.py#L79); original repo: [nnzhan/Graph-WaveNet](https://github.com/nnzhan/Graph-WaveNet)
- AGCRN's variant of the same construction: [`src/models/agcrn.py#L95`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/models/agcrn.py#L95); original repo: [LeiBAI/AGCRN](https://github.com/LeiBAI/AGCRN)
- tsl LargeST loader (Hugging Face URLs, `key="readings"`, district mask, edge-list adjacency): [`tsl/datasets/large_st.py`](https://github.com/TorchSpatiotemporal/tsl/blob/c46a23ee1b95674cfcbb00a60cecf9c2ad06c9bc/tsl/datasets/large_st.py) (L11, L165–L168, L206, L213–L219, L229–L233)

In [ ]:
# ---------------- switches ----------------
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')  # deterministic cuBLAS; must be set before CUDA starts

SMOKE_TEST = False   # True: synthetic data in the real file format, 2 epochs -> checks the whole pipeline quickly
USE_DRIVE  = False   # True: cache processed data + results in Google Drive (MyDrive/DLSTG_PA3)
RESUME     = True    # reuse finished models (checkpoint + metrics) instead of retraining
SEED       = 42
AGG        = 'sum'   # 15-min aggregation of 5-min readings: 'sum' (interval vehicle counts) or 'mean' (rates); see Q3(a)

CFG = dict(
    seed=SEED, in_len=8, out_len=4,           # 2 h history -> 1 h horizon at 15 min
    hidden=32, kernel=3,                      # temporal encoder width / kernel
    emb_dim=10, topk=10, alpha=0.5,           # adaptive graph: node-embedding size, kept neighbours, hybrid mix
    lr=1e-3, weight_decay=0.0, batch_size=64, grad_clip=5.0,
    max_epochs=50, patience=5,                # early stopping on validation MAE (cap keeps one run under ~1 h on a T4)
    loss='masked MAE on per-sensor z-scored flow',
)
if SMOKE_TEST:
    CFG.update(max_epochs=2, patience=1)

In [ ]:
import sys, json, time, math, random, shutil, gc, glob, zipfile, platform, subprocess, importlib.util, urllib.request, warnings
from fractions import Fraction

for pkg in ('tables', 'hdf5plugin'):   # pandas HDF5 support; compression filters (e.g. Blosc) so h5py can read compressed files
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
try:
    import hdf5plugin                  # registers the filters with h5py on import
except ImportError:
    print('hdf5plugin unavailable: compressed files may need the slower pandas fallback')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/DLSTG_PA3'
elif IN_COLAB:
    BASE = '/content/pa3'
else:
    BASE = os.path.abspath('pa3_run')
RAW_DIR = '/content/largest_raw' if IN_COLAB else os.path.join(BASE, 'raw')   # big raw files: never on Drive
if SMOKE_TEST:
    BASE += '_smoke'; RAW_DIR += '_smoke'
DATA = os.path.join(BASE, 'data')                       # processed San Diego data, reused if it already exists
RUN = os.path.join(BASE, 'run_residual')               # outputs of this model version (older checkpoints are never resumed)
RES, FIG, CKPT = (os.path.join(RUN, d) for d in ('results', 'figures', 'checkpoints'))
for d in (RAW_DIR, RES, FIG, CKPT, DATA):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)   # warns (does not fail) for ops without a deterministic kernel
set_seed()

def save_fig(name):
    plt.savefig(os.path.join(FIG, name + '.png'), dpi=150, bbox_inches='tight')
    plt.show()

def save_json(obj, name):
    with open(os.path.join(RES, name), 'w') as f:
        json.dump(obj, f, indent=2, default=str)

print('Colab:', IN_COLAB, '| device:', DEVICE, '| output folder:', RUN, '| smoke test:', SMOKE_TEST)

In [ ]:
# Hardware and software record for Q7
def hardware_info():
    info = dict(platform=platform.platform(), python=platform.python_version(), torch=torch.__version__,
                numpy=np.__version__, pandas=pd.__version__, h5py=h5py.__version__,
                cpu_count=os.cpu_count(), device=str(DEVICE), colab=IN_COLAB)
    try:
        with open('/proc/cpuinfo') as f:
            info['cpu_model'] = next(l.split(':', 1)[1].strip() for l in f if l.startswith('model name'))
    except Exception:
        info['cpu_model'] = platform.processor()
    try:
        with open('/proc/meminfo') as f:
            info['ram_gb'] = round(int(f.readline().split()[1]) / 1024**2, 1)
    except Exception:
        pass
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info['gpu'] = p.name
        info['gpu_memory_gb'] = round(p.total_memory / 1024**3, 1)
    return info

HW = hardware_info()
save_json(HW, 'hardware.json')
HW

## Question 1 — What does a sum remember?

### Q1(a) Affine message
$\sum_{u}\phi(x_u) = w S_v + c\, n_v$, so two neighbourhoods with equal $(n_v, S_v)$ give the same aggregate for every $w, c$, and $\rho$ must give the same output. Example: $\{-1,-1,+1,+1\}$ (0 steady) and $\{-1,0,0,+1\}$ (2 steady), both $n_v=4$, $S_v=0$.

In [ ]:
N1, N2 = [-1, -1, +1, +1], [-1, 0, 0, +1]
for nb in (N1, N2):
    print(nb, '| n_v =', len(nb), '| S_v =', sum(nb), '| steady =', nb.count(0))
rng = np.random.default_rng(SEED)
for _ in range(5):
    w, c = rng.normal(size=2)
    agg = [sum(w * x + c for x in nb) for nb in (N1, N2)]
    print(f'w={w:+.3f}  c={c:+.3f}  ->  aggregate N1 = {agg[0]:+.6f},  N2 = {agg[1]:+.6f}')
    assert np.isclose(agg[0], agg[1])

### Q1(b) A scalar message that encodes all three counts
Base-$B$ digits with $B = K+1$: $\phi(-1)=1,\ \phi(0)=B,\ \phi(+1)=B^2$. The sum is $q = n_- + B n_0 + B^2 n_+$; each count is at most $K < B$, so there are no carries. Decode: $n_- = q \bmod B$, $n_0 = \lfloor q/B \rfloor \bmod B$, $n_+ = \lfloor q/B^2 \rfloor$. For $K=8$: $\phi = (1, 9, 81)$. The check below is the one from the solution guide.

Non-steady count only: $\phi(-1)=1,\ \phi(0)=0,\ \phi(+1)=1$, whose sum is $n_- + n_+$.

In [ ]:
from itertools import product

K = 8
B = K + 1
triples = np.array([counts for counts in product(range(K + 1), repeat=3) if 1 <= sum(counts) <= K], dtype=int)

values = triples @ np.array([1, B, B**2])
assert len(values) == 164
assert len(np.unique(values)) == len(values)
recovered = np.column_stack((values % B, (values // B) % B, values // (B**2)))
assert np.array_equal(recovered, triples)
print(f'phi(-1), phi(0), phi(+1) = 1, {B}, {B**2}:  {len(values)} neighbourhoods, all sums distinct, all decoded correctly')

nonsteady = triples @ np.array([1, 0, 1])
assert np.array_equal(nonsteady, triples[:, 0] + triples[:, 2])
print('phi = (1, 0, 1): the sum equals the number of non-steady neighbours for every neighbourhood')

### Q1(c) Sum, mean and max with one-hot messages
Exhaustive check over every neighbourhood of size 1–8. A method can solve a task exactly when no two neighbourhoods share an aggregated vector but have different answers. For every "No" the code reports the first (smallest) such pair; for every "Yes" it checks the stated rule on all 164 neighbourhoods. Means are kept as exact fractions.

In [ ]:
nbhds = [(nm, n0, size - nm - n0) for size in range(1, K + 1) for nm in range(size + 1) for n0 in range(size - nm + 1)]
AGGS = {
    'Sum':  lambda c: tuple(c),
    'Mean': lambda c: tuple(Fraction(x, sum(c)) for x in c),
    'Max':  lambda c: tuple(int(x > 0) for x in c),
}
TASKS = {
    '(i) count steady':            lambda c: c[1],
    '(ii) more than half slowing': lambda c: c[0] > sum(c) / 2,
    '(iii) any slowing':           lambda c: c[0] > 0,
}
YES_RULES = {  # rule rho applies to the aggregated vector v = (v_-, v_0, v_+)
    ('(i) count steady', 'Sum'):            ('read v_0 (= n_0)',                         lambda v: v[1]),
    ('(ii) more than half slowing', 'Sum'): ('v_- > (v_- + v_0 + v_+)/2',                lambda v: v[0] > sum(v) / 2),
    ('(ii) more than half slowing', 'Mean'):('v_- > 1/2',                                 lambda v: v[0] > Fraction(1, 2)),
    ('(iii) any slowing', 'Sum'):           ('v_- > 0',                                   lambda v: v[0] > 0),
    ('(iii) any slowing', 'Mean'):          ('v_- > 0',                                   lambda v: v[0] > 0),
    ('(iii) any slowing', 'Max'):           ('v_- == 1',                                  lambda v: v[0] == 1),
}
show = lambda c: '{' + ','.join(['-1'] * c[0] + ['0'] * c[1] + ['+1'] * c[2]) + '}'
fmt = lambda v: '(' + ', '.join(str(x) for x in v) + ')'

rows = []
for tname, task in TASKS.items():
    for aname, agg in AGGS.items():
        first_seen, witness = {}, None
        for c in nbhds:                      # ordered by size, so the first collision is a smallest example
            v = agg(c)
            if v in first_seen and task(first_seen[v]) != task(c):
                witness = (first_seen[v], c, v); break
            first_seen.setdefault(v, c)
        if witness is None:
            rule_txt, rule = YES_RULES[(tname, aname)]
            assert all(rule(agg(c)) == task(c) for c in nbhds)
            rows.append(dict(task=tname, method=aname, possible='Yes', reason=f'rho: {rule_txt}'))
        else:
            a, b, v = witness
            rows.append(dict(task=tname, method=aname, possible='No',
                             reason=f'{show(a)} and {show(b)} both give {fmt(v)}; answers {task(a)} vs {task(b)}'))
assert len(rows) == len(YES_RULES) + 3
q1c = pd.DataFrame(rows)
q1c.to_csv(os.path.join(RES, 'q1c_table.csv'), index=False)
pd.set_option('display.max_colwidth', 120)
q1c

## Question 2 — Which way is the slowdown moving?
Ring of 40 sensors, true rule $x_i(t+1) = x_{i+1}(t)$, i.e. `np.roll(x, -1)`. With `np.roll(x, 1)[i] = x[i-1]` and `np.roll(x, -1)[i] = x[i+1]`.

### Q2(a)(i) Least-squares fit of the symmetric layer

In [ ]:
NR = 40
x = np.zeros(NR); x[20] = 1.0
target = np.roll(x, -1)
feats = np.stack([x, np.roll(x, 1) + np.roll(x, -1)], axis=1)     # columns: x_i, x_{i-1} + x_{i+1}
(a_fit, b_fit), *_ = np.linalg.lstsq(feats, target, rcond=None)
pred = feats @ np.array([a_fit, b_fit])
print(f'a = {a_fit:.6f}, b = {b_fit:.6f}   (analytic: a = 0, b = 1/2)')
print('sensors 19, 20, 21  prediction:', pred[19:22].round(3), ' target:', target[19:22])
print('sum of squared errors:', round(float(((pred - target) ** 2).sum()), 6))

### Q2(a)(ii) Iterating the fitted layer vs the true shift

In [ ]:
i_ring = np.arange(NR)
bump = np.exp(-(i_ring - 20) ** 2 / 8)
fit_seq, true_seq = [bump], [bump]
for _ in range(20):
    xf = fit_seq[-1]
    fit_seq.append(a_fit * xf + b_fit * (np.roll(xf, 1) + np.roll(xf, -1)))
    true_seq.append(np.roll(true_seq[-1], -1))
fit_seq, true_seq = np.array(fit_seq), np.array(true_seq)
vmin, vmax = 0.0, max(fit_seq.max(), true_seq.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, seq, title in zip(axes, (fit_seq, true_seq), ('Fitted symmetric layer', 'Exact backward shift')):
    im = ax.imshow(seq, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title); ax.set_xlabel('Sensor index')
axes[0].set_ylabel('Time step t')
fig.colorbar(im, ax=axes, label='Slowdown value')
save_fig('q2a_ring_heatmaps')

com = lambda s: float((s * i_ring).sum() / s.sum())
print(f't=20  fitted: peak {fit_seq[-1].max():.3f}, centre of mass {com(fit_seq[-1]):.2f}   |   '
      f'true: peak {true_seq[-1].max():.3f}, centre of mass {com(true_seq[-1]):.2f}')

### Q2(b) Direction-specific weights and the 1-D convolution
$a=0,\ w_-=0,\ w_+=1$. As a convolution over $[x_{i-1}, x_i, x_{i+1}]$ the kernel is $[0, 0, 1]$ with circular padding (PyTorch `Conv1d` is a cross-correlation, so the weights apply in that order).

In [ ]:
a_b, w_minus, w_plus = 0.0, 0.0, 1.0
rng = np.random.default_rng(SEED)
for _ in range(100):
    xr = rng.normal(size=NR)
    assert np.allclose(a_b * xr + w_minus * np.roll(xr, 1) + w_plus * np.roll(xr, -1), np.roll(xr, -1))

conv = nn.Conv1d(1, 1, kernel_size=3, padding=1, padding_mode='circular', bias=False)
with torch.no_grad():
    conv.weight[:] = torch.tensor([[[w_minus, a_b, w_plus]]])
xt = torch.tensor(xr, dtype=torch.float32).view(1, 1, -1)
assert torch.allclose(conv(xt).view(-1), torch.tensor(np.roll(xr, -1), dtype=torch.float32), atol=1e-6)
print('Directional layer and circular Conv1d with kernel [0, 0, 1] reproduce the next snapshot exactly (100 random inputs).')

### Q2(c) Adding the previous snapshot
$a=0,\ b=1,\ c=-1$. The same coefficients are checked for backward and forward motion, on random snapshots and on a 20-step rollout of the bump.

In [ ]:
a_c, b_c, c_c = 0.0, 1.0, -1.0
hist_layer = lambda xt_, xprev: a_c * xt_ + b_c * (np.roll(xt_, 1) + np.roll(xt_, -1)) + c_c * xprev

rng = np.random.default_rng(SEED)
for name, step in (('backward x_i(t+1)=x_{i+1}(t)', -1), ('forward  x_i(t+1)=x_{i-1}(t)', +1)):
    errs = []
    for _ in range(100):
        xprev = rng.normal(size=NR)
        xcur = np.roll(xprev, step)
        errs.append(np.abs(hist_layer(xcur, xprev) - np.roll(xcur, step)).max())
    print(f'{name}: max |error| over 100 random pairs = {max(errs):.2e}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (title, step) in zip(axes, (('Backward motion', -1), ('Forward motion', +1))):
    prev, cur, seq = np.roll(bump, -step), bump, [bump]
    for _ in range(20):
        prev, cur = cur, hist_layer(cur, prev)
        seq.append(cur)
    expected = np.array([np.roll(bump, step * t) for t in range(21)])
    assert np.allclose(np.array(seq), expected)
    im = ax.imshow(np.array(seq), aspect='auto', cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'History layer (a,b,c)=(0,1,-1): {title}'); ax.set_xlabel('Sensor index')
axes[0].set_ylabel('Time step t')
fig.colorbar(im, ax=axes, label='Slowdown value')
save_fig('q2c_history_layer_both_directions')

## Question 3 — Dataset preparation and graph construction

### Q3(a) Download, San Diego extraction and 15-minute resampling
**Sensors.** San Diego = metadata rows with `district == 11` (716 sensors), the rule used by both the LargeST notebook and the tsl loader. The same ordered index list is applied to the readings and to both axes of the adjacency.

**Reading.** Each yearly file holds all 8,600 sensors at 5-minute resolution under key `readings`. To keep RAM low, only the 716 San Diego columns are read, in row blocks with `h5py`. If the file layout is not the expected pandas "fixed" format, the code falls back to `pandas.read_hdf` (which loads the whole year first). Each year is reindexed to the full 5-minute calendar, so missing timestamps become explicit missing values.

**Units and aggregation.** PeMS 5-minute "flow" is a vehicle count per 5-minute interval, summed over lanes. That is a secondary-source description, so the code below also prints the 5-minute value ranges as a plausibility check. Following the solution guide's rule for counts: the 15-minute value is the **sum** of the three 5-minute counts (units: vehicles per 15 minutes) and is only formed when all three are observed; otherwise it is missing. LargeST's own code averages instead (`resample('15T').mean()`), so our errors are ≈3× the LargeST paper's scale and are not directly comparable to it.

**Missing values.** Missing 15-minute values stay `NaN` here. In Q4, model *inputs* are filled with the sensor's training mean (0 after z-scoring), and missing *targets* are masked out of the loss and all metrics. Zeros are kept as genuine readings; their count is reported.

In [ ]:
HF_BASE = 'https://huggingface.co/datasets/TorchSpatiotemporal/ProcessedDatasets/resolve/v1.0.0/large_st/'
YEARS = [2020, 2021]
EXPECTED_SD = 716
PAPER_SD_EDGES = 17319          # LargeST paper, SD subset (off-diagonal nonzeros)
PROC = os.path.join(DATA, 'sd_processed.npz')

def fetch(url, dest):
    if os.path.exists(dest):
        print('found', dest); return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print('downloading', url); t0 = time.time()
    tmp = dest + '.part'
    with urllib.request.urlopen(url) as r, open(tmp, 'wb') as f:
        shutil.copyfileobj(r, f, length=16 << 20)
    os.replace(tmp, dest)
    print(f'  {os.path.getsize(dest) / 1e6:.0f} MB in {time.time() - t0:.0f} s')
    return dest

def make_synthetic_raw(raw_dir, n_total=100, n_sd=60, seed=SEED):
    """SMOKE_TEST only: tiny files in the same layout as the real ones (metadata.csv, adj.npz, YEAR/data.h5)."""
    rng = np.random.default_rng(seed)
    sens = os.path.join(raw_dir, 'sensors'); os.makedirs(sens, exist_ok=True)
    district = np.r_[np.full(n_sd, 11), rng.choice([4, 7, 12], n_total - n_sd)]
    rng.shuffle(district)
    lat, lng = 32.6 + rng.uniform(0, 0.5, n_total), -117.3 + rng.uniform(0, 0.5, n_total)
    meta = pd.DataFrame({'lat': lat, 'lng': lng, 'district': district},
                        index=pd.Index(np.arange(400000, 400000 + n_total), name='id'))
    meta.to_csv(os.path.join(sens, 'metadata.csv'))
    d = np.hypot(lat[:, None] - lat[None], lng[:, None] - lng[None])
    rows, cols, wts = [], [], []
    for i in range(n_total):
        for j in np.argsort(d[i])[:6]:                       # itself + 5 nearest, directed
            rows.append(i); cols.append(j); wts.append(np.exp(-(d[i, j] / 0.05) ** 2))
    np.savez(os.path.join(sens, 'adj.npz'), edge_index=np.array([rows, cols]), edge_weight=np.array(wts, np.float32))
    for y in YEARS:
        idx = pd.date_range(f'{y}-01-01', f'{y + 1}-01-01', freq='5min', inclusive='left')
        tod = ((idx.hour * 60 + idx.minute).values / 1440.0)[:, None]
        vals = (40 + 30 * np.sin(2 * np.pi * (tod - 0.3))) * rng.uniform(0.5, 1.5, n_total) \
               + rng.normal(0, 3, (len(idx), n_total))
        vals = vals.clip(0).round().astype(np.float32)
        vals[rng.random(vals.shape) < 0.02] = np.nan
        df = pd.DataFrame(vals, index=idx, columns=meta.index.to_numpy())
        df = df.drop(df.index[rng.choice(len(df), 50, replace=False)])     # some missing timestamps
        os.makedirs(os.path.join(raw_dir, str(y)), exist_ok=True)
        df.to_hdf(os.path.join(raw_dir, str(y), 'data.h5'), key='readings', mode='w')
    print('synthetic raw files written to', raw_dir)

def find_file(root, name):
    hits = glob.glob(os.path.join(root, '**', name), recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name} not found under {root}')
    return hits[0]

def h5_column_labels(path, key='readings'):
    try:
        with h5py.File(path, 'r') as f:
            ax0 = f[key]['axis0'][:]
        return np.array([a.decode() if isinstance(a, bytes) else str(a) for a in ax0])
    except Exception as e:
        print('  could not read column labels with h5py:', e)
        return None

def read_columns(path, cols, year, key='readings'):
    """Read only the given column positions -> (DatetimeIndex, float32 array [rows, len(cols)])."""
    cols = np.asarray(cols)
    order = np.argsort(cols); cols_sorted = cols[order]; inv = np.argsort(order)
    try:
        with h5py.File(path, 'r') as f:
            g = f[key]
            if 'block0_values' not in g or 'block1_values' in g:
                raise ValueError('not a single-block pandas fixed-format frame')
            if not np.array_equal(g['axis0'][:], g['block0_items'][:]):
                raise ValueError('block items are not in column order')
            v, ax1 = g['block0_values'], g['axis1']
            n_rows, n_items = ax1.shape[0], g['block0_items'].shape[0]
            if v.shape == (n_rows, n_items):           # pandas stores the frame transposed: rows x columns
                out = np.empty((n_rows, len(cols)), np.float32)
                for s in range(0, n_rows, 4096):
                    out[s:s + 4096] = v[s:s + 4096][:, cols_sorted]
            elif v.shape == (n_items, n_rows):
                out = v[cols_sorted.tolist(), :].T.astype(np.float32)
            else:
                raise ValueError(f'unexpected value shape {v.shape}')
            raw_idx = ax1[:]
            tz = ax1.attrs.get('tz', None)
        if tz is not None:
            tz = tz.decode() if isinstance(tz, bytes) else str(tz)
            index = pd.to_datetime(raw_idx, utc=True).tz_convert(tz).tz_localize(None)
        else:
            index = pd.to_datetime(raw_idx)
        if (index.year == year).mean() < 0.9:
            raise ValueError('time index did not decode to the expected year')
        print('  read with h5py (only the selected columns)')
        return index, out[:, inv]
    except Exception as e:
        print(f'  h5py fast path unavailable ({e}); falling back to pandas.read_hdf (loads the whole year)')
        df = pd.read_hdf(path, key=key)
        out = df.iloc[:, cols].to_numpy(np.float32)
        index = pd.DatetimeIndex(df.index)
        del df; gc.collect()
        return index, out

In [ ]:
if os.path.exists(PROC):
    print('processed San Diego data found:', PROC)
else:
    # ---- 1. obtain raw files ----
    sens_dir = os.path.join(RAW_DIR, 'sensors')
    if SMOKE_TEST:
        if not glob.glob(os.path.join(RAW_DIR, '*', 'data.h5')):
            make_synthetic_raw(RAW_DIR)
    else:
        if not glob.glob(os.path.join(sens_dir, '**', 'metadata.csv'), recursive=True):
            zpath = fetch(HF_BASE + 'sensors.zip', os.path.join(RAW_DIR, 'sensors.zip'))
            with zipfile.ZipFile(zpath) as z:
                z.extractall(sens_dir)
        for y in YEARS:
            fetch(HF_BASE + f'{y}/data.h5', os.path.join(RAW_DIR, str(y), 'data.h5'))

    # ---- 2. metadata and San Diego selection ----
    meta_all = pd.read_csv(find_file(sens_dir, 'metadata.csv'), index_col=0)
    lower = {c.lower(): c for c in meta_all.columns}
    sd_pos = np.flatnonzero((meta_all[lower['district']] == 11).values)
    meta_sd = meta_all.iloc[sd_pos].copy()
    sd_ids = np.array([str(i) for i in meta_sd.index], dtype=str)   # plain unicode array, loads without pickle
    print(f'metadata: {len(meta_all)} sensors, {len(sd_pos)} in district 11 (San Diego)')
    if not SMOKE_TEST:
        assert len(sd_pos) == EXPECTED_SD, f'expected {EXPECTED_SD} San Diego sensors, found {len(sd_pos)}'

    # ---- 3. adjacency: edge list over metadata rows -> 716 x 716 in the same order ----
    z = np.load(find_file(sens_dir, 'adj.npz'))
    if 'edge_index' in z.files:
        ei, ew = z['edge_index'], z['edge_weight']
    else:
        ei, ew = (z[k] for k in z.files)
    remap = np.full(len(meta_all), -1); remap[sd_pos] = np.arange(len(sd_pos))
    r, c = remap[ei[0]], remap[ei[1]]
    keep = (r >= 0) & (c >= 0)
    A_sd = np.zeros((len(sd_pos), len(sd_pos)), np.float32)
    A_sd[r[keep], c[keep]] = ew[keep]

    # ---- 4. readings: same sensor order ----
    first = os.path.join(RAW_DIR, str(YEARS[0]), 'data.h5')
    labels = h5_column_labels(first)
    if labels is not None and set(sd_ids) <= set(labels):
        lookup = {s: i for i, s in enumerate(labels)}
        read_pos = np.array([lookup[s] for s in sd_ids])
        print('columns matched by sensor ID; same as metadata row order:', np.array_equal(read_pos, sd_pos))
    else:
        read_pos = sd_pos
        print('column labels are not the metadata IDs -> selecting by metadata row position (as the tsl loader does)')

    stats, flows, times = {}, [], []
    for y in YEARS:
        print(f'year {y}:')
        idx, vals = read_columns(os.path.join(RAW_DIR, str(y), 'data.h5'), read_pos, y)
        full = pd.date_range(f'{y}-01-01', f'{y + 1}-01-01', freq='5min', inclusive='left')
        dfy = pd.DataFrame(vals, index=idx)
        n_dup = int(dfy.index.duplicated().sum())
        dfy = dfy[~dfy.index.duplicated()].sort_index()
        on_grid = dfy.index.isin(full)
        dfy = dfy[on_grid].reindex(full)
        v5 = dfy.to_numpy(np.float32)
        obs5 = ~np.isnan(v5)
        st = dict(rows_in_file=len(idx), duplicate_timestamps=n_dup, off_grid_rows=int((~on_grid).sum()),
                  expected_5min_rows=len(full), missing_timestamps=int(len(full) - on_grid.sum()),
                  missing_5min_values=int((~obs5).sum()), total_5min_values=int(v5.size),
                  zero_5min_values=int((v5 == 0).sum()), negative_5min_values=int((v5 < 0).sum()),
                  q50_5min=float(np.nanpercentile(v5, 50)), q99_5min=float(np.nanpercentile(v5, 99)),
                  max_5min=float(np.nanmax(v5)))
        v3 = v5.reshape(-1, 3, v5.shape[1])
        if AGG == 'sum':
            v15 = v3.sum(axis=1)                       # NaN if any of the three is missing -> all three required
        else:
            cnt = (~np.isnan(v3)).sum(axis=1)
            v15 = np.where(cnt >= 2, np.nansum(v3, axis=1) / np.maximum(cnt, 1), np.nan).astype(np.float32)
        st.update(missing_15min_values=int(np.isnan(v15).sum()), total_15min_values=int(v15.size))
        stats[str(y)] = st
        print('  ', st)
        flows.append(v15.astype(np.float32)); times.append(full[::3])
        del vals, dfy, v5, v3; gc.collect()

    flow15 = np.concatenate(flows); times15 = times[0].append(times[1:])
    np.savez_compressed(PROC, flow=flow15, times=np.asarray(times15.values, dtype='datetime64[ns]'),
                        A=A_sd, sd_ids=sd_ids, sd_pos=np.asarray(sd_pos, dtype=np.int64))
    meta_sd.to_csv(os.path.join(DATA, 'sd_meta.csv'))
    summary = dict(source=HF_BASE, years=YEARS, n_sensors=len(sd_pos), aggregation=AGG,
                   units='vehicles per 15 min (sum of three 5-min counts)' if AGG == 'sum'
                         else 'vehicles per 5 min, averaged over the 15-min block',
                   per_year=stats,
                   missing_5min_total=sum(s['missing_5min_values'] for s in stats.values()),
                   missing_15min_total=sum(s['missing_15min_values'] for s in stats.values()))
    save_json(summary, 'q3a_data_summary.json')
    shutil.copy(os.path.join(RES, 'q3a_data_summary.json'), DATA)   # kept with the processed data for later runs
    del flows, flow15; gc.collect()
    # The raw files are no longer needed; on Colab they live on the temporary disk anyway.

In [ ]:
def load_times(arr):
    """Saved timestamps -> DatetimeIndex. Also decodes integer timestamps written by an earlier version."""
    if np.issubdtype(arr.dtype, np.datetime64):
        return pd.DatetimeIndex(arr)
    for unit in ('ns', 'us', 'ms', 's'):
        t = pd.to_datetime(arr, unit=unit)
        if t[0].year == YEARS[0]:
            return t
    raise ValueError('could not decode the saved timestamps; delete ' + PROC + ' and rerun the previous cell')

Z = np.load(PROC, allow_pickle=True)          # our own file; pickle only needed for caches from an earlier version
flow, times = Z['flow'], load_times(Z['times'])
A_raw, sd_ids = Z['A'], np.array([str(s) for s in Z['sd_ids']], dtype=str)
assert times[0] == pd.Timestamp(f'{YEARS[0]}-01-01') and times[-1] == pd.Timestamp(f'{YEARS[-1]}-12-31 23:45'), (times[0], times[-1])
assert (np.diff(times.values) == np.timedelta64(15, 'm')).all(), 'timestamps are not a regular 15-minute grid'
assert flow.shape == (len(times), len(sd_ids)) and A_raw.shape == (len(sd_ids), len(sd_ids))
meta_sd = pd.read_csv(os.path.join(DATA, 'sd_meta.csv'), index_col=0)
T, N = flow.shape
OBS = ~np.isnan(flow)
lower = {c.lower(): c for c in meta_sd.columns}
lat_col = next((lower[k] for k in ('lat', 'latitude') if k in lower), None)
lng_col = next((lower[k] for k in ('lng', 'lon', 'long', 'longitude') if k in lower), None)
LAT = meta_sd[lat_col].to_numpy(float) if lat_col else None
LNG = meta_sd[lng_col].to_numpy(float) if lng_col else None

q3a_path = next(p for p in (os.path.join(RES, 'q3a_data_summary.json'), os.path.join(DATA, 'q3a_data_summary.json'),
                            os.path.join(BASE, 'results', 'q3a_data_summary.json')) if os.path.exists(p))
with open(q3a_path) as f:
    q3a = json.load(f)
save_json(q3a, 'q3a_data_summary.json')
print(f'flow tensor: T={T} 15-min steps x N={N} sensors; {times[0]} -> {times[-1]}')
print(f'observed 15-min values: {OBS.mean():.2%}; sensors with no observations at all: {int((~OBS.any(0)).sum())}')
pd.DataFrame(q3a['per_year']).T

### Q3(b) The road-distance graph
LargeST builds edges from road-network driving distances between sensors, weighted by a Gaussian kernel $\exp(-d_{ij}^2/\sigma^2)$ with small weights removed, so a nonzero $A_{ij}$ means sensors $i$ and $j$ are close along the road network, with larger weights for shorter distances. The row convention used throughout is $\mathcal N(i)=\{j : A_{ij}\neq 0\}$ (row $i$ lists the sensors that node $i$ aggregates from).

In [ ]:
A_off = A_raw.copy(); np.fill_diagonal(A_off, 0)
Bphys = A_off != 0
out_deg, in_deg = Bphys.sum(1), Bphys.sum(0)
und = Bphys | Bphys.T
deg_und = und.sum(1)
symmetric_values = bool(np.allclose(A_raw, A_raw.T))
symmetric_pattern = bool(np.array_equal(Bphys, Bphys.T))
q3b = {
    'nodes': int(N),
    'nonzero entries (incl. diagonal)': int(np.count_nonzero(A_raw)),
    'diagonal nonzero entries': int(np.count_nonzero(np.diag(A_raw))),
    'off-diagonal nonzero entries (directed edges)': int(Bphys.sum()),
    'LargeST paper SD edge count (reference)': PAPER_SD_EDGES,
    'unique undirected edges (i<j, either direction)': int(np.triu(und, 1).sum()),
    'symmetric (values, allclose)': symmetric_values,
    'symmetric (sparsity pattern)': symmetric_pattern,
    'entries with A_ij != A_ji': int((~np.isclose(A_off, A_off.T)).sum()),
    'weight min / max (nonzero, off-diag)': [float(A_off[Bphys].min()), float(A_off[Bphys].max())] if Bphys.any() else None,
    'out-degree mean / min / max': [float(out_deg.mean()), int(out_deg.min()), int(out_deg.max())],
    'in-degree mean / min / max': [float(in_deg.mean()), int(in_deg.min()), int(in_deg.max())],
    'undirected degree mean / min / max': [float(deg_und.mean()), int(deg_und.min()), int(deg_und.max())],
    'isolated sensors (no edge either way)': int((deg_und == 0).sum()),
}
save_json(q3b, 'q3b_graph_summary.json')
pd.Series(q3b, name='value').to_frame()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw=dict(width_ratios=[1.1, 1]))
im = axes[0].imshow(np.ma.masked_equal(A_off, 0), cmap='viridis', interpolation='nearest')
axes[0].set_title('San Diego road-distance adjacency (diagonal removed;\nwhite = 0), same order as the flow tensor')
axes[0].set_xlabel('sensor j'); axes[0].set_ylabel('sensor i')
fig.colorbar(im, ax=axes[0], fraction=0.046, label='weight $A_{ij}$')
bins = np.arange(0, max(out_deg.max(), in_deg.max()) + 2) - 0.5
if symmetric_pattern:
    axes[1].hist(out_deg, bins=bins, color='tab:blue')
else:
    axes[1].hist([out_deg, in_deg], bins=bins, label=['out-degree (row nonzeros)', 'in-degree (column nonzeros)'])
    axes[1].legend()
axes[1].set_xlabel('number of neighbours'); axes[1].set_ylabel('number of sensors')
axes[1].set_title('Neighbours per sensor')
save_fig('q3b_adjacency_and_degree')

**Sensors used for examples.** Chosen from the graph alone, before any model is trained: among sensors with at least 90% observed 15-minute values, the one with the highest undirected degree, the median one, and the lowest one with at least one neighbour. The same three sensors are used in the Q3(a) week plot, Q3(c) (median one) and every Q6 prediction plot.

In [ ]:
eligible = np.flatnonzero((OBS.mean(0) >= 0.9) & (deg_und > 0))
if len(eligible) < 3:
    eligible = np.flatnonzero(deg_und > 0)
order_ = eligible[np.argsort(deg_und[eligible], kind='stable')]
S_HI, S_MED, S_LO = int(order_[-1]), int(order_[len(order_) // 2]), int(order_[0])
EXAMPLE = {'high degree': S_HI, 'median degree': S_MED, 'low degree': S_LO}
for k, v in EXAMPLE.items():
    print(f'{k:14s}: position {v:3d}, sensor ID {sd_ids[v]}, undirected degree {deg_und[v]}, observed {OBS[:, v].mean():.1%}')

week0 = pd.Timestamp('2020-03-02')            # a Monday, before the COVID-19 stay-at-home order
p0 = times.get_loc(week0)
wk = slice(p0, p0 + 7 * 96)
fig, ax = plt.subplots(figsize=(13, 4))
for k, v in EXAMPLE.items():
    ax.plot(times[wk], flow[wk, v], lw=1, label=f'sensor {sd_ids[v]} ({k}, {int((~OBS[wk, v]).sum())} missing)')
ax.set_ylabel('flow (vehicles / 15 min)' if AGG == 'sum' else 'flow (vehicles / 5 min, 15-min mean)')
ax.set_title('15-minute traffic flow, one week (2–8 March 2020); gaps are missing values')
ax.legend(fontsize=8)
save_fig('q3a_one_week_three_sensors')

### Q3(c) One-hop and two-hop neighbourhoods

In [ ]:
i0 = S_MED
one_hop = np.flatnonzero(Bphys[i0])
reach2 = Bphys[i0] | ((Bphys[i0].astype(np.float32) @ Bphys.astype(np.float32)) > 0)
reach2[i0] = False
within_two = np.flatnonzero(reach2)
exactly_two = np.setdiff1d(within_two, one_hop)
q3c = dict(sensor_position=i0, sensor_id=sd_ids[i0], convention='N(i) = {j : A_ij != 0}, self excluded',
           one_hop_ids=sd_ids[one_hop].tolist(), within_two_hops_ids=sd_ids[within_two].tolist(),
           exactly_two_hops_ids=sd_ids[exactly_two].tolist(),
           counts=dict(one_hop=len(one_hop), within_two=len(within_two), exactly_two=len(exactly_two)))
save_json(q3c, 'q3c_neighbourhoods.json')
print(f"sensor {sd_ids[i0]}: {len(one_hop)} one-hop, {len(exactly_two)} exactly-two-hop, {len(within_two)} within two hops")
print('one-hop IDs:', q3c['one_hop_ids'])
print('exactly-two-hop IDs:', q3c['exactly_two_hops_ids'])

if LAT is not None:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(LNG, LAT, s=4, c='lightgrey', label='all SD sensors')
    ax.scatter(LNG[exactly_two], LAT[exactly_two], s=18, c='tab:orange', label='exactly two hops')
    ax.scatter(LNG[one_hop], LAT[one_hop], s=22, c='tab:blue', label='one hop')
    ax.scatter(LNG[i0], LAT[i0], s=90, c='red', marker='*', label=f'sensor {sd_ids[i0]}')
    ax.set_xlabel('longitude'); ax.set_ylabel('latitude'); ax.legend(fontsize=8)
    ax.set_title('One- and two-hop neighbourhoods')
    save_fig('q3c_neighbourhood_map')

## Question 4 — Spatio-temporal forecasting

### Q4(a) Windows, chronological splits and normalisation
Splits: train = 2020, validation = Jan–Mar 2021, test = Apr–Dec 2021. Windows are built inside each split, so no input or target crosses a boundary: $X_t=[x_{t-7},\dots,x_t]$, $Y_t=[x_{t+1},\dots,x_{t+4}]$ (offsets as in LargeST's `generate_data_for_training.py`). Windows with no observed target at all are dropped. The same test windows and masks are used for every model.

Normalisation is a per-sensor z-score fitted only on observed 2020 values. A sensor with no training observations gets the global training mean and standard deviation; a standard deviation below 1e-6 is replaced with 1. Missing inputs become 0 after z-scoring (the sensor's training mean).

In [ ]:
def bounds(mask):
    idx = np.flatnonzero(mask)
    assert len(idx) and np.all(np.diff(idx) == 1), 'split is not contiguous'
    return int(idx[0]), int(idx[-1]) + 1

SPLITS = {
    'train': bounds(times < '2021-01-01'),
    'val':   bounds((times >= '2021-01-01') & (times < '2021-04-01')),
    'test':  bounds(times >= '2021-04-01'),
}
L_IN, L_OUT = CFG['in_len'], CFG['out_len']
row_any = OBS.any(1)
def window_starts(s, e):
    t = np.arange(s + L_IN - 1, e - L_OUT)          # x_{t-7} >= s and x_{t+4} <= e-1
    has_target = np.zeros(len(t), bool)
    for h in range(1, L_OUT + 1):
        has_target |= row_any[t + h]
    return t[has_target]
WIN = {k: window_starts(*v) for k, v in SPLITS.items()}

tr0, tr1 = SPLITS['train']
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    MU = np.nanmean(flow[tr0:tr1], axis=0)
    SD = np.nanstd(flow[tr0:tr1], axis=0)
no_train = np.isnan(MU)
MU[no_train] = np.nanmean(flow[tr0:tr1]); SD[no_train] = np.nanstd(flow[tr0:tr1])
small_sd = ~(SD > 1e-6)
SD[small_sd] = 1.0
MU, SD = MU.astype(np.float32), SD.astype(np.float32)

Xn = np.where(OBS, (flow - MU) / SD, 0.0).astype(np.float32)
RAW0 = np.where(OBS, flow, 0.0).astype(np.float32)

X_t = torch.tensor(Xn, device=DEVICE)
M_t = torch.tensor(OBS, device=DEVICE)
R_t = torch.tensor(RAW0, device=DEVICE)
MU_t, SD_t = torch.tensor(MU, device=DEVICE), torch.tensor(SD, device=DEVICE)
WIN_t = {k: torch.tensor(v, device=DEVICE, dtype=torch.long) for k, v in WIN.items()}
OFF_IN = torch.arange(-L_IN + 1, 1, device=DEVICE)
OFF_OUT = torch.arange(1, L_OUT + 1, device=DEVICE)

def get_batch(t):
    ti, to = t[:, None] + OFF_IN, t[:, None] + OFF_OUT
    return X_t[ti], X_t[to], M_t[to], R_t[to]      # (B,8,N) input, (B,4,N) target z, mask, target raw

split_info = {k: dict(first=str(times[s]), last=str(times[e - 1]), steps=e - s, windows=int(len(WIN[k])))
              for k, (s, e) in SPLITS.items()}
split_info['normalisation'] = dict(kind='per-sensor z-score, observed 2020 values only',
                                   sensors_without_train_data=int(no_train.sum()), sensors_sd_replaced=int(small_sd.sum()))
save_json(split_info, 'q4a_splits.json')
pd.DataFrame({k: v for k, v in split_info.items() if k != 'normalisation'}).T

### Q4(b) Model
1. **Temporal encoder** (shared by all sensors): `Conv1d(1→d, k=3)` → ReLU → `Conv1d(d→d, k=3, dilation 2)` → ReLU → flatten the 2 remaining time steps → `Linear(2d→d)` → ReLU, giving $Z\in\mathbb R^{B\times N\times d}$.
2. **Graph convolution with a residual connection**: $H = Z + \mathrm{ReLU}(\hat A Z W_g + b_g)$, a batched matrix product over the node dimension. The residual keeps each sensor's own temporal features and lets the graph term add neighbour information on top, as Graph WaveNet does in every layer ([`gwnet.py#L88`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/models/gwnet.py#L88), [`#L105`](https://github.com/liuxu77/LargeST/blob/b7c6ff5f7f0c10051ed10a319b2cfe58021312f9/src/models/gwnet.py#L105)).
   *Why:* an earlier run without the residual, $H=\mathrm{ReLU}(\hat A Z W_g + b_g)$, gave a test MAE of 93.7 for the fixed graph against 61.5 for the temporal-only model. Under row normalisation a sensor's own features get weight $1/(\text{out-degree}+1)$, about 4% at the mean out-degree of 24, so its forecast was driven mostly by its neighbours' average. The adaptive graph, whose self-weight is 0.5 after adding $I$, matched the baseline, and the hybrid fell in between, which is consistent with that explanation.
3. **Head**: `Linear(d→4)` per node, reshaped to $(B, 4, N)$.

Training: Adam, masked MAE on z-scored values, gradient clipping, early stopping on validation MAE in original units; the best-validation weights are restored before testing. **Training time** is wall-clock for the training loop including the per-epoch validation pass; it excludes data preparation and the final test evaluation.

**Fixed-graph normalisation (Q5a).** The diagonal is cleared and self-loops are added once. If the road graph is symmetric we use $D^{-1/2}(A+I)D^{-1/2}$; otherwise the row normalisation $D^{-1}(A+I)$, which keeps the row convention ("row $i$ aggregates its neighbours") without imposing symmetry. These correspond to LargeST's `calculate_sym_adj` and `calculate_asym_adj`.

In [ ]:
def normalize_adj(A, mode):
    A = A + torch.eye(A.shape[0], device=A.device, dtype=A.dtype)
    d = A.sum(1)
    if mode == 'sym':
        dinv = d.pow(-0.5)
        return dinv[:, None] * A * dinv[None, :]
    return A / d[:, None]

FIXED_NORM = 'sym' if symmetric_values else 'row'
A_FIXED_HAT = normalize_adj(torch.tensor(A_off, device=DEVICE), FIXED_NORM)
print('fixed-graph normalisation:', FIXED_NORM)

class TemporalEncoder(nn.Module):
    def __init__(self, d, k, t_in):
        super().__init__()
        self.conv1 = nn.Conv1d(1, d, k)
        self.conv2 = nn.Conv1d(d, d, k, dilation=2)
        t_left = t_in - (k - 1) - 2 * (k - 1)
        assert t_left >= 1
        self.proj = nn.Linear(d * t_left, d)

    def forward(self, x):                              # x: (B, T_in, N)
        b, t, n = x.shape
        h = x.permute(0, 2, 1).reshape(b * n, 1, t)    # each sensor's history is one sequence
        h = F.relu(self.conv2(F.relu(self.conv1(h))))
        return F.relu(self.proj(h.flatten(1))).view(b, n, -1)

class GraphConv(nn.Module):
    """H = Z + ReLU(A_hat Z W_g + b_g): graph convolution with a residual connection (as in Graph WaveNet)."""
    def __init__(self, d, residual=True):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
        self.residual = residual

    def forward(self, Z, A_hat):                       # (N,N) @ (B,N,d) -> (B,N,d)
        H = F.relu(torch.matmul(A_hat, self.W(Z)) + self.b)
        return Z + H if self.residual else H

class FixedAdj(nn.Module):
    def __init__(self, A_hat):
        super().__init__()
        self.register_buffer('A_hat', A_hat.clone())

    def forward(self):
        return self.A_hat

class STModel(nn.Module):
    """adj=None -> temporal-only baseline (same encoder and head, no graph layer)."""
    def __init__(self, adj=None, d=CFG['hidden'], k=CFG['kernel'], t_in=L_IN, t_out=L_OUT):
        super().__init__()
        self.enc = TemporalEncoder(d, k, t_in)
        self.adj = adj
        self.gcn = GraphConv(d) if adj is not None else None
        self.head = nn.Linear(d, t_out)

    def forward(self, x):
        z = self.enc(x)
        h = self.gcn(z, self.adj()) if self.adj is not None else z
        return self.head(h).permute(0, 2, 1)          # (B, T_out, N)

In [ ]:
def masked_mae(pred, y, m):
    m = m.float()
    return (torch.abs(pred - y) * m).sum() / m.sum().clamp_min(1.0)

@torch.no_grad()
def evaluate(model, split, bs=256):
    """MAE / RMSE in original flow units over observed targets only (overall and per horizon step)."""
    model.eval()
    ts = WIN_t[split]
    abs_h = torch.zeros(L_OUT, dtype=torch.float64, device=DEVICE)
    sq_h, cnt_h = abs_h.clone(), abs_h.clone()
    for s in range(0, len(ts), bs):
        x, _, m, r = get_batch(ts[s:s + bs])
        p = model(x) * SD_t + MU_t
        e = (p - r) * m
        abs_h += e.abs().sum((0, 2)).double(); sq_h += (e ** 2).sum((0, 2)).double(); cnt_h += m.sum((0, 2)).double()
    return dict(mae=float(abs_h.sum() / cnt_h.sum()), rmse=float(torch.sqrt(sq_h.sum() / cnt_h.sum())),
                mae_per_step=(abs_h / cnt_h).tolist(), rmse_per_step=torch.sqrt(sq_h / cnt_h).tolist(),
                n_targets=int(cnt_h.sum()))

def run_experiment(name, build):
    mpath, cpath = os.path.join(RES, f'metrics_{name}.json'), os.path.join(CKPT, f'{name}.pt')
    set_seed(SEED)
    model = build().to(DEVICE)
    if RESUME and os.path.exists(mpath) and os.path.exists(cpath):
        model.load_state_dict(torch.load(cpath, map_location=DEVICE))
        with open(mpath) as f:
            metrics = json.load(f)
        print(f'[{name}] loaded finished run: test MAE {metrics["test"]["mae"]:.3f}, RMSE {metrics["test"]["rmse"]:.3f}')
        return model, metrics
    opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    gen = torch.Generator().manual_seed(SEED)
    train_ts, bs = WIN_t['train'], CFG['batch_size']
    hist, best, best_state, best_ep, bad = [], float('inf'), None, 0, 0
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for ep in range(1, CFG['max_epochs'] + 1):
        model.train()
        perm = train_ts[torch.randperm(len(train_ts), generator=gen).to(DEVICE)]
        tot, nb = torch.zeros((), device=DEVICE), 0
        for s in range(0, len(perm), bs):
            x, y, m, _ = get_batch(perm[s:s + bs])
            loss = masked_mae(model(x), y, m)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            opt.step()
            tot += loss.detach(); nb += 1
        val = evaluate(model, 'val')
        elapsed = time.perf_counter() - t0
        hist.append(dict(epoch=ep, train_loss_z=float(tot) / nb, val_mae=val['mae'], val_rmse=val['rmse'], elapsed_s=elapsed))
        print(f'[{name}] epoch {ep:2d}  train loss (z) {float(tot) / nb:.4f}  val MAE {val["mae"]:.3f}  '
              f'val RMSE {val["rmse"]:.3f}  ({elapsed:.0f}s)')
        if val['mae'] < best - 1e-4:
            best, best_ep, bad = val['mae'], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG['patience']:
                break
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    train_time = time.perf_counter() - t0
    model.load_state_dict(best_state)
    metrics = dict(name=name, best_epoch=best_ep, epochs_run=len(hist), train_time_s=train_time,
                   time_per_epoch_s=train_time / len(hist), n_params=sum(p.numel() for p in model.parameters()),
                   val=evaluate(model, 'val'), test=evaluate(model, 'test'))
    torch.save(model.state_dict(), cpath)
    save_json(hist, f'history_{name}.json')
    save_json(metrics, f'metrics_{name}.json')
    print(f'[{name}] best epoch {best_ep}: val MAE {metrics["val"]["mae"]:.3f} | test MAE {metrics["test"]["mae"]:.3f}, '
          f'RMSE {metrics["test"]["rmse"]:.3f} | training {train_time:.0f}s')
    return model, metrics

MODELS, METRICS = {}, {}

### Q4(b) + Q5(a) STGNN with the fixed road graph, and Q4(c) temporal-only baseline
The temporal-only baseline uses the same encoder, head, windows, normalisation, seed, optimiser, stopping rule and evaluation mask; only the graph layer is removed.

In [ ]:
EXPERIMENTS = {'temporal_only': lambda: STModel(None),
               'stgnn_fixed':   lambda: STModel(FixedAdj(A_FIXED_HAT))}
for name in ('temporal_only', 'stgnn_fixed'):
    MODELS[name], METRICS[name] = run_experiment(name, EXPERIMENTS[name])

## Question 5 — Learnable and hybrid adjacency

### Q5(b) Adaptive adjacency
$E_1, E_2\in\mathbb R^{N\times d_e}$ ($d_e=10$, initialised `randn` as in Graph WaveNet), $A_{\text{adaptive}}=\mathrm{softmax}_{\text{row}}(\mathrm{ReLU}(E_1E_2^\top))$, learned only from the training loss.

**Order of operations:** scores $E_1E_2^\top$ → ReLU → row softmax → keep the top $k=10$ entries per row → renormalise each row to sum to 1 → add self-loops and row-normalise, $D^{-1}(A+I)$, the same convention as the fixed graph when that graph is directed.

**Gradients:** `topk` chooses the index set without a gradient. For the chosen entries, gradients flow through the kept probabilities and the renormalisation back to $E_1, E_2$; dropped entries get no gradient on that step. The kept set is recomputed every forward pass, so it can change during training. A row whose scores are all ≤ 0 becomes uniform after softmax and gives no gradient through the ReLU on that pass.

### Q5(c) Hybrid adjacency
$\hat A_h=\alpha\hat A_f+(1-\alpha)\hat A_a$ with fixed $\alpha=0.5$. Both parts are normalised operators that include self-loops. When the road graph is directed both are row-normalised; if it turns out symmetric, the fixed part is symmetric-normalised and the adaptive part stays row-normalised because a learned graph is directed by construction.

In [ ]:
class AdaptiveAdj(nn.Module):
    def __init__(self, n, de=CFG['emb_dim'], k=CFG['topk']):
        super().__init__()
        self.E1 = nn.Parameter(torch.randn(n, de))
        self.E2 = nn.Parameter(torch.randn(n, de))
        self.k = k

    def probs(self):                                  # dense softmax(relu(E1 E2^T)), row-wise
        return torch.softmax(F.relu(self.E1 @ self.E2.T), dim=1)

    def sparse(self):                                 # top-k per row, renormalised
        P = self.probs()
        if self.k is None or self.k >= P.shape[1]:
            return P
        val, idx = P.topk(self.k, dim=1)
        S = torch.zeros_like(P).scatter(1, idx, val)
        return S / S.sum(1, keepdim=True)

    def forward(self):
        return normalize_adj(self.sparse(), 'row')

class HybridAdj(nn.Module):
    def __init__(self, A_hat_fixed, n, alpha=CFG['alpha']):
        super().__init__()
        self.fixed = FixedAdj(A_hat_fixed)
        self.adaptive = AdaptiveAdj(n)
        self.alpha = alpha

    def forward(self):
        return self.alpha * self.fixed() + (1 - self.alpha) * self.adaptive()

EXPERIMENTS.update(stgnn_adaptive=lambda: STModel(AdaptiveAdj(N)),
                   stgnn_hybrid=lambda: STModel(HybridAdj(A_FIXED_HAT, N)))
for name in ('stgnn_adaptive', 'stgnn_hybrid'):
    MODELS[name], METRICS[name] = run_experiment(name, EXPERIMENTS[name])

In [ ]:
# Learned adjacency (Q5b item 4): top-k operator next to the road graph, same sensor order
@torch.no_grad()
def learned_graph(model):
    adj = model.adj.adaptive if isinstance(model.adj, HybridAdj) else model.adj
    return adj.probs().cpu().numpy(), adj.sparse().cpu().numpy()

LEARNED = {name: learned_graph(MODELS[name]) for name in ('stgnn_adaptive', 'stgnn_hybrid')}
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
panels = [('Road-distance graph (binary)', Bphys.astype(float)),
          ('Adaptive model: top-10 learned graph', LEARNED['stgnn_adaptive'][1]),
          ('Hybrid model: learned (adaptive) part', LEARNED['stgnn_hybrid'][1])]
for ax, (title, M) in zip(axes, panels):
    im = ax.imshow(np.ma.masked_equal(M, 0), cmap='viridis', interpolation='nearest')
    ax.set_title(title); ax.set_xlabel('sensor j'); ax.set_ylabel('sensor i')
    fig.colorbar(im, ax=ax, fraction=0.046)
save_fig('q5b_learned_adjacency')

for name, (P, S) in LEARNED.items():
    rows_ = np.repeat(np.arange(N), CFG['topk'])
    cols_ = np.argsort(-S, axis=1)[:, :CFG['topk']].ravel()
    pd.DataFrame(dict(source_pos=rows_, target_pos=cols_, source_id=sd_ids[rows_], target_id=sd_ids[cols_],
                      weight=S[rows_, cols_])).to_csv(os.path.join(RES, f'learned_edges_{name}.csv'), index=False)

## Question 6 — Experimental analysis

### Q6(a) Comparison table (test set, original units, best-validation checkpoint)

In [ ]:
LABELS = {'temporal_only': ('Temporal-only', 'None'), 'stgnn_fixed': ('STGNN', 'Fixed'),
          'stgnn_adaptive': ('STGNN', 'Adaptive'), 'stgnn_hybrid': ('STGNN', 'Hybrid')}
table = pd.DataFrame([dict(Model=LABELS[k][0], Adjacency=LABELS[k][1],
                           MAE=round(m['test']['mae'], 3), RMSE=round(m['test']['rmse'], 3),
                           **{'Training time (s)': round(m['train_time_s'], 1)},
                           **{'Val MAE': round(m['val']['mae'], 3), 'Val RMSE': round(m['val']['rmse'], 3),
                              'Best / run epochs': f"{m['best_epoch']} / {m['epochs_run']}",
                              's per epoch': round(m['time_per_epoch_s'], 1), 'Parameters': m['n_params']})
                      for k, m in METRICS.items()])
table.to_csv(os.path.join(RES, 'q6a_results_table.csv'), index=False)
with open(os.path.join(RES, 'q6a_results_table.md'), 'w') as f:
    f.write(table.to_markdown(index=False) if importlib.util.find_spec('tabulate') else table.to_string(index=False))

per_step = pd.DataFrame({LABELS[k][1] if k != 'temporal_only' else 'Temporal-only':
                         [f"{a:.2f} / {r:.2f}" for a, r in zip(m['test']['mae_per_step'], m['test']['rmse_per_step'])]
                         for k, m in METRICS.items()}, index=[f'+{15 * h} min' for h in range(1, L_OUT + 1)])
per_step.index.name = 'test MAE / RMSE'
per_step.to_csv(os.path.join(RES, 'q6a_per_horizon.csv'))
display(table)
display(per_step)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k in METRICS:
    with open(os.path.join(RES, f'history_{k}.json')) as f:
        h = pd.DataFrame(json.load(f))
    lab = ' '.join(LABELS[k])
    axes[0].plot(h.epoch, h.train_loss_z, label=lab)
    axes[1].plot(h.epoch, h.val_mae, label=lab)
axes[0].set_title('Training loss (masked MAE, z-scored)'); axes[1].set_title('Validation MAE (original units)')
for ax in axes:
    ax.set_xlabel('epoch'); ax.legend(fontsize=8)
save_fig('q6a_training_curves')

In [ ]:
# Predicted vs actual: same 3 sensors, same 2 test days, horizons +15 min and +60 min, all models
@torch.no_grad()
def predict(model, t_arr, bs=256):
    model.eval(); out = []
    t_all = torch.tensor(t_arr, device=DEVICE, dtype=torch.long)
    for s in range(0, len(t_all), bs):
        x, _, _, _ = get_batch(t_all[s:s + bs])
        out.append((model(x) * SD_t + MU_t).cpu().numpy())
    return np.concatenate(out)                           # (len, 4, N)

day0 = times.get_loc(pd.Timestamp('2021-04-05'))          # first Monday of the test period
tau = np.arange(day0, day0 + 2 * 96)                      # target times
t_needed = np.arange(tau[0] - L_OUT, tau[-1])             # prediction origins t = tau - h
assert t_needed[0] >= SPLITS['test'][0] + L_IN - 1
PRED = {k: predict(m, t_needed) for k, m in MODELS.items()}

fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex=True)
for r, (lab_s, i) in enumerate(EXAMPLE.items()):
    for c, h in enumerate((1, L_OUT)):
        ax = axes[r, c]
        ax.plot(times[tau], flow[tau, i], color='black', lw=2, label='actual')
        for k, P in PRED.items():
            ax.plot(times[tau], P[tau - h - t_needed[0], h - 1, i], lw=1, label=' '.join(LABELS[k]))
        ax.set_title(f'sensor {sd_ids[i]} ({lab_s}), +{15 * h} min ahead', fontsize=10)
        if c == 0:
            ax.set_ylabel('flow')
axes[0, 0].legend(fontsize=8)
fig.autofmt_xdate()
save_fig('q6a_pred_vs_actual')

### Q6(b) Learned graph vs road graph
For each sensor $i$: $L_k(i)$ = its learned top-10 neighbours and $P(i)$ = its road-graph neighbours (row convention), **self-loops excluded from both**. Precision@10 $=|L\cap P|/|L|$, Jaccard $=|L\cap P|/|L\cup P|$. Geographic distances use the haversine formula on the metadata latitude/longitude ($R=6371$ km). "Within two road hops" checks whether a learned edge links sensors that are at most two edges apart in the road graph. Random sensor pairs give a reference distance.

In [ ]:
def haversine_km(i, j):
    la1, la2 = np.radians(LAT[i]), np.radians(LAT[j])
    dla, dlo = la2 - la1, np.radians(LNG[j] - LNG[i])
    return 2 * 6371.0088 * np.arcsin(np.sqrt(np.sin(dla / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin(dlo / 2) ** 2))

two_hop_phys = Bphys | ((Bphys.astype(np.float32) @ Bphys.astype(np.float32)) > 0)   # float -> BLAS, much faster than int
np.fill_diagonal(two_hop_phys, False)
rng = np.random.default_rng(SEED)
ri, rj = rng.integers(0, N, 20000), rng.integers(0, N, 20000)
ri, rj = ri[ri != rj], rj[ri != rj]

q6b_rows, DIST = [], {}
if LAT is not None:
    pi, pj = np.nonzero(Bphys)
    DIST['road-graph edges'] = haversine_km(pi, pj)
    DIST['random pairs'] = haversine_km(ri, rj)
for name, (P, S) in LEARNED.items():
    L = S > 0
    self_mass = float(np.diag(S).mean())
    np.fill_diagonal(L, False)
    inter, lsz, union = (L & Bphys).sum(1), L.sum(1), (L | Bphys).sum(1)
    row = {'model': ' '.join(LABELS[name]),
           'mean Precision@10': float((inter / np.maximum(lsz, 1)).mean()),
           'mean Jaccard@10': float((inter / np.maximum(union, 1)).mean()),
           'learned edges that are road edges': float((L & Bphys).sum() / max(L.sum(), 1)),
           'learned edges within 2 road hops': float((L & two_hop_phys).sum() / max(L.sum(), 1)),
           'mean learned self-weight (top-k matrix diagonal)': self_mass,
           'max in-degree of learned graph': int(L.sum(0).max()),
           'sensors that are nobody\'s learned neighbour': int((L.sum(0) == 0).sum())}
    if LAT is not None:
        li, lj = np.nonzero(L)
        DIST[f'learned edges ({LABELS[name][1].lower()})'] = haversine_km(li, lj)
        row['mean distance learned edges (km)'] = float(DIST[f'learned edges ({LABELS[name][1].lower()})'].mean())
        row['median distance learned edges (km)'] = float(np.median(DIST[f'learned edges ({LABELS[name][1].lower()})']))
        row['mean distance road edges (km)'] = float(DIST['road-graph edges'].mean())
        row['mean distance random pairs (km)'] = float(DIST['random pairs'].mean())
    q6b_rows.append(row)
q6b = pd.DataFrame(q6b_rows).set_index('model').T
q6b.to_csv(os.path.join(RES, 'q6b_graph_comparison.csv'))
q6b

In [ ]:
if LAT is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    bins = np.linspace(0, np.percentile(np.concatenate(list(DIST.values())), 99), 60)
    for lab, d in DIST.items():
        axes[0].hist(d, bins=bins, density=True, histtype='step', lw=1.5, label=f'{lab} (n={len(d)})')
    axes[0].set_xlabel('haversine distance (km)'); axes[0].set_ylabel('density'); axes[0].legend(fontsize=8)
    axes[0].set_title('Distance between connected sensors')

    P, S = LEARNED['stgnn_adaptive']
    S_ = S.copy(); np.fill_diagonal(S_, 0)
    top = np.argsort(-S_, axis=None)[:300]
    ti, tj = np.unravel_index(top, S_.shape)
    from matplotlib.collections import LineCollection
    ax = axes[1]
    seg = lambda a, b: np.stack([np.c_[LNG[a], LAT[a]], np.c_[LNG[b], LAT[b]]], axis=1)
    pi, pj = np.nonzero(Bphys)
    ax.add_collection(LineCollection(seg(pi, pj), colors='lightgrey', linewidths=0.5, zorder=1))
    ax.add_collection(LineCollection(seg(ti, tj), colors=np.where(Bphys[ti, tj], 'tab:green', 'tab:red').tolist(),
                                     linewidths=0.9, alpha=0.8, zorder=2))
    ax.scatter(LNG, LAT, s=3, c='black', zorder=3)
    ax.plot([], [], color='lightgrey', label='road-graph edges')
    ax.plot([], [], color='tab:green', label='strongest learned edges that are road edges')
    ax.plot([], [], color='tab:red', label='strongest learned edges not in the road graph')
    ax.legend(fontsize=8, loc='lower left'); ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
    ax.set_title('300 strongest learned edges (adaptive model)')
    save_fig('q6b_learned_vs_road_graph')
else:
    print('No latitude/longitude columns found in the metadata; skipping the geographic analysis.')

### Q6(c) Critical discussion
This part is written in the report, using the numbers above.

## Question 7 — Reproducibility
The cell below writes `results/config.json`. `hardware.json`, per-model `history_*.json` and `metrics_*.json` were written above. The last cell packs everything for download.

In [ ]:
config = dict(
    CFG, aggregation=AGG, smoke_test=SMOKE_TEST, device=str(DEVICE),
    data=dict(source=HF_BASE, files=['sensors.zip'] + [f'{y}/data.h5' for y in YEARS], licence='CC BY-NC 4.0',
              subset='district == 11 (San Diego)', n_sensors=int(N), sensor_order='metadata row order (sd_meta.csv)',
              resampling='15-min sum of three 5-min counts, all three required' if AGG == 'sum'
                         else '15-min mean of 5-min values, at least two required'),
    splits=split_info,
    missing_values='inputs: sensor training mean (0 after z-score); targets: masked from loss and metrics',
    graph_layer='H = Z + ReLU(A_hat Z W_g + b_g) (residual), head on H; temporal-only: head on Z',
    fixed_adjacency=dict(diagonal='cleared, then +I once', normalisation=FIXED_NORM),
    adaptive_adjacency='softmax_row(relu(E1 E2^T)) -> top-k per row -> row renormalise -> +I -> row normalise',
    hybrid=f"alpha * fixed_hat + (1 - alpha) * adaptive_hat, alpha = {CFG['alpha']} (not learned)",
    optimiser='Adam', early_stopping=f"validation MAE, patience {CFG['patience']}, best weights restored",
    training_time='wall-clock of training loop incl. per-epoch validation; excludes data prep and test evaluation',
    determinism='seeded python/numpy/torch, cudnn.deterministic, use_deterministic_algorithms(warn_only=True)',
    example_sensors={k: dict(position=v, id=sd_ids[v]) for k, v in EXAMPLE.items()},
)
save_json(config, 'config.json')
print(json.dumps(config, indent=2, default=str)[:3000])

In [ ]:
# Pack results/, figures/, checkpoints/ into one zip and download it (processed data is not included)
zip_path = os.path.join(os.path.dirname(BASE), 'pa3_outputs' + ('_smoke' if SMOKE_TEST else '') + '.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for sub in ('results', 'figures', 'checkpoints'):
        for p in sorted(glob.glob(os.path.join(RUN, sub, '*'))):
            zf.write(p, arcname=os.path.join(sub, os.path.basename(p)))
print(f'{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB')
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)